# Topic 25 - pgvector (hand-written semantics stand-in)

Companion notebook for `pgvector_sim.py`. A faithful miniature of pgvector: typed vector columns with the same dimension errors, the `<->` / `<=>` / `<#>` operators, exact Seq Scan vs ANN Index Scan, IVFFlat (k-means lists + probes) and HNSW (layered graph + ef_search), filters applied after the index scan (plus pgvector 0.8-style iterative scans), and recall monitoring - all offline and seeded, so every run reproduces.

## Learning objectives

- see the topic's mechanics running offline, cell by cell
- connect each demo cell to the pytest suite that pins it
- walk away with numbers you can quote from memory

## 0. Import the core

In [1]:
import math

from pgvector_sim import (
    HNSWIndex, IVFFlatIndex, VectorTable, build_corpus,
    cosine_distance, index_health_check, kmeans, l2_distance,
    negative_inner_product, recall, storage_bytes,
)


## 1. The vector column

`vector(dims)` validates dimensions at insert (an error, never coercion); storage is 4 x dims + 8 bytes.

In [2]:
table, queries = build_corpus(n_per_cluster=10, dimensions=3, seed=7)
print(f"table rows: {len(table)}  dims: {table.dimensions}")
for dims in (384, 1536):
    print(f"  vector({dims}) storage = {storage_bytes(dims)} bytes/row")
try:
    table.add("bad", [1.0, 2.0])  # wrong dims
except ValueError as exc:
    print(f"insert with wrong dims -> {exc}")


table rows: 50  dims: 3
  vector(384) storage = 1544 bytes/row
  vector(1536) storage = 6152 bytes/row
insert with wrong dims -> different vector dimensions 2 and 3


## 2. The operator mirrors

`<->` L2, `<=>` cosine distance, `<#>` negative inner product (zero vectors give NaN - exactly like pgvector).

In [3]:
q = [1.0, 2.0]
for vec, label in [([2, 4], "same direction"), ([-2, 1], "orthogonal"), ([-1, -2], "opposite")]:
    print(f"{label:16s}  <-> {l2_distance(q, vec):6.3f}   <=> {cosine_distance(q, vec):6.3f}   <#> {negative_inner_product(q, vec):7.3f}")
print(f"zero vector     <=> is NaN? {math.isnan(cosine_distance([0, 0], q))}")


same direction    <->  2.236   <=>  0.000   <#> -10.000
orthogonal        <->  3.162   <=>  1.000   <#>  -0.000
opposite          <->  4.472   <=>  2.000   <#>   5.000
zero vector     <=> is NaN? True


## 3. Exact search = Seq Scan

Every row scanned, sorted ascending by distance; perfect recall, O(n).

In [4]:
q0 = queries[0]
exact = table.search(q0, op="<=>", k=5)
print(table.plan())
for row, d in exact:
    print(f"  id={row.id:2d} cluster={row.data['cluster']}  distance={d:.4f}")


Seq Scan on items
  id= 8 cluster=0  distance=0.0009
  id= 0 cluster=0  distance=0.0018
  id= 9 cluster=0  distance=0.0049
  id= 6 cluster=0  distance=0.0075
  id= 3 cluster=0  distance=0.0079


## 4. IVFFlat: lists built by seeded k-means, widened by probes

lists=15 partition 50 docs; probes=1 (pgvector's default!) visibly drops neighbours; probes=list-count is exact search again.

In [5]:
ivf = IVFFlatIndex(table, lists=15, seed=42).build()
print(ivf.plan())
print(f"k-means settled after {ivf.km.iterations} Lloyd iterations")
print(f"lists sizes = {[len(v) for v in ivf.lists_rows.values()]}")
print(f"\n{'probes':>7s} {'recall@5':>9s}")
for probes in (1, 2, 4, 15):
    scores = [recall(ivf.search(q, k=5, probes=probes), table.search(q, k=5)) for q in queries]
    print(f"{probes:7d} {sum(scores) / len(scores):9.2f}")


Index Scan using items_ivfflat_idx
k-means settled after 3 Lloyd iterations
lists sizes = [3, 1, 4, 7, 6, 4, 1, 3, 4, 4, 1, 3, 3, 3, 3]

 probes  recall@5
      1      0.64
      2      0.96
      4      1.00
     15      1.00


## 5. HNSW: layered graph, ef_search as the runtime knob

No training step; build inserts each point into the graph; search = greedy descent + beam of ef candidates. Higher ef -> higher recall (Crunchy Data: keep m and ef_construction modest).

In [6]:
h = HNSWIndex(table, m=8, ef_construction=64, seed=42).build()
print(h.plan())
print(f"max layer = {h.max_layer}, entry point = row {h.entry_point}")
print(f"{'ef_search':>9s} {'recall@5':>9s}")
for ef in (4, 8, 16, 40):
    scores = [recall(h.search(q, k=5, ef_search=ef), table.search(q, k=5)) for q in queries]
    print(f"{ef:9d} {sum(scores) / len(scores):9.2f}")


Index Scan using items_hnsw_idx
max layer = 2, entry point = row 19
ef_search  recall@5
        4      1.00
        8      1.00
       16      1.00
       40      1.00


## 6. Filters: AFTER the index scan (and the 0.8 fix)

With ef_search=12 the beam holds 12 candidates; a filter matching a sliver of rows passes only a couple - fewer than LIMIT asked for. The iterative scan keeps widening until enough rows match (pgvector 0.8+ behaviour).

In [7]:
q0 = queries[0]
hits = h.search(q0, k=3, ef_search=12, where={"cluster": 0})
print(f"post-filtered with default scan: {len(hits)} of 3 requested")
hits = h.search(q0, k=3, ef_search=12, where={"cluster": 0}, iterative=True)
print(f"iterative scan (0.8+ behaviour):  {len(hits)} of 3")
exact_cluster0 = table.search(q0, k=3, where={"cluster": 0})
assert [r.id for r, _ in hits] == [r.id for r, _ in exact_cluster0]
print("\niterative results match the exact filtered search")


post-filtered with default scan: 3 of 3 requested
iterative scan (0.8+ behaviour):  3 of 3

iterative results match the exact filtered search


## 7. Monitor recall like the docs suggest

Compare each index against exact on your own queries (pgvector does it with `SET LOCAL enable_indexscan = off`).

In [8]:
scores = index_health_check(table, queries, k=5)
for key, score in scores.items():
    print(f"  {key:20s} recall@5 = {score:.2f}")


  ivfflat_probes_1     recall@5 = 0.92
  ivfflat_probes_3     recall@5 = 1.00
  ivfflat_probes_5     recall@5 = 1.00
  hnsw_ef_search_16    recall@5 = 1.00
  hnsw_ef_search_40    recall@5 = 1.00


## 8. Run the tests

In [9]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-1500:])
assert "failed" not in result.stdout, "pytest failures!"
print("\noffline tests passed")


......................................                                   [100%]
38 passed in 0.25s


offline tests passed


## 9. What to notice

- vector columns are typed: wrong dims raise, like real postgres
- `<#>` is negative inner product because Postgres scans ascending
- IVFFlat recall is monotonic in probes; probes=1 (the default) is a recall hole
- HNSW trades memory + build time for better speed-recall; ef_search is the knob
- filters run after the index scan: use iterative scans / partial indexes
- monitor recall against exact exactly as the pgvector docs prescribe